Начнём с подготовки данных с помощью Pandas:

Очистим данные от выбросов - более 1000 покупок в день - это могут быть боты. Оставляем в датафрейме только тех пользователей, которые не являются ботами.

In [1]:
import pandas as pd

df = pd.read_csv('2019-Nov.csv')

df['event_time'] = pd.to_datetime(df['event_time'])
df['date'] = df['event_time'].dt.date


purchases = df[df['event_type'] == 'purchase']


daily_purchases = purchases.groupby(['user_id', 'date']).size().reset_index(name='purchase_count')

bot_users = daily_purchases[daily_purchases['purchase_count'] > 1000]['user_id'].unique()

df_filtered = df[~df['user_id'].isin(bot_users)].copy()

df_filtered = df_filtered.drop(columns=['date'])

Нам понадобятся только целевые события view -> cart -> purchase - то есть только те товары, которые положили в корзину и затем купили. Дополнительно зачистим артефакты, которые были куплены, но предварительно не попали в корзину.

In [2]:
events_to_keep = ['view', 'cart', 'purchase']
df_funnel = df_filtered[df_filtered['event_type'].isin(events_to_keep)].copy()

df_funnel = df_funnel.sort_values(by=['user_session', 'product_id', 'event_time'])

df_funnel['is_cart'] = (df_funnel['event_type'] == 'cart').astype(int)

df_funnel['cart_cumsum'] = df_funnel.groupby(['user_session', 'product_id'])['is_cart'].cumsum()

anomaly_purchases_mask = (df_funnel['event_type'] == 'purchase') & (df_funnel['cart_cumsum'] == 0)

df_clean = df_funnel[~anomaly_purchases_mask].drop(columns=['is_cart', 'cart_cumsum'])

df_clean = df_clean.sort_values(by='event_time').reset_index(drop=True)

Теперь рассчитаем макро-конверсию ((Overall CR): % дошедших до покупки от зашедших в приложение) и микро-конверсии ((Step-to-step CR): % перехода из просмотра в корзину и из корзины в оплату.)

In [3]:
funnel_stages = df_clean.groupby('event_type')['user_id'].nunique().reindex(['view', 'cart', 'purchase'])

users_view = funnel_stages['view']
users_cart = funnel_stages['cart']
users_purchase = funnel_stages['purchase']

macro_cr = (users_purchase / users_view) * 100

cr_view_to_cart = (users_cart / users_view) * 100
cr_cart_to_purchase = (users_purchase / users_cart) * 100

metrics_df = pd.DataFrame({
    'Stage': ['view', 'cart', 'purchase'],
    'Unique_Users': [users_view, users_cart, users_purchase],
    'Step_CR_%': [100.0, cr_view_to_cart, cr_cart_to_purchase],
    'Overall_CR_%': [100.0, (users_cart / users_view) * 100, macro_cr]
})

print(metrics_df.round(2))

      Stage  Unique_Users  Step_CR_%  Overall_CR_%
0      view        355643     100.00        100.00
1      cart         95939      26.98         26.98
2  purchase         25571      26.65          7.19


По данному выводу визуализируем воронку

In [4]:
import plotly.graph_objects as go

stages_labels = ['Просмотр - View', 'Корзина - Cart', 'Покупка - Purchase']

fig = go.Figure(go.Funnel(
    y=stages_labels,
    x=metrics_df['Unique_Users'],
    textposition="inside",
    textinfo="value+percent previous+percent initial",
    opacity=0.85,
    marker={
        "color": ["#4C78A8", "#F58518", "#54A24B"],
        "line": {"width": [2, 2, 2], "color": ["white", "white", "white"]}
    },
    connector={"line": {"color": "royalblue", "dash": "dot", "width": 2}}
))

fig.update_layout(
    title="Воронка конверсии мобильного приложения",
    title_x=0.5, 
    template="plotly_white",
    width=800,
    height=500
)

fig.show()

In [5]:
fig.write_image("funnel_chart.png", scale=2)

В ячейке ниже рассчитаем метрику Time-to-purchase - медианное время от первого просмотра карточки товара до его оплаты. Это необходимо для понимания пользовательского поведения: например, если мы приняли решение изменить UX сайта, а TTP увеличился, это может указывать на неэффективность нового дизайна. 

Также, отдаем предпочтение медиане, поскольку среднее сильно искажается редкими выбросами.

In [6]:
first_views = df_clean[df_clean['event_type'] == 'view'].groupby(['user_session', 'product_id'])['event_time'].min().reset_index(name='first_view_time')
purchases = df_clean[df_clean['event_type'] == 'purchase'].groupby(['user_session', 'product_id'])['event_time'].min().reset_index(name='purchase_time')

time_data = pd.merge(first_views, purchases, on=['user_session', 'product_id'], how='inner')


time_data['time_to_purchase'] = time_data['purchase_time'] - time_data['first_view_time']

median_ttp = time_data['time_to_purchase'].median()

print(f"Медианное время от первого просмотра до покупки: {median_ttp}")

Медианное время от первого просмотра до покупки: 0 days 00:16:24


##### Переходим к этапу сегментации

Будем сегментировать по времени суток - в ячейке ниже сравним поведение пользователей днём в промежуток 6:00-18:00 и ночью в промежуток 18:00-6:00

In [8]:
df_clean['hour'] = df_clean['event_time'].dt.hour
df_clean['time_segment'] = df_clean['hour'].apply(
    lambda x: 'День (06:00-18:00)' if 6 <= x < 18 else 'Ночь (18:00-06:00)'
)

funnel_segmented = df_clean.groupby(['time_segment', 'event_type'])['user_id'].nunique().unstack()

funnel_segmented = funnel_segmented[['view', 'cart', 'purchase']] 

funnel_segmented['macro_cr_%'] = (funnel_segmented['purchase'] / funnel_segmented['view'] * 100).round(2)

print(funnel_segmented)

event_type            view   cart  purchase  macro_cr_%
time_segment                                           
День (06:00-18:00)  263578  72964     18014        6.83
Ночь (18:00-06:00)  142621  41037      8662        6.07


In [9]:
import plotly.graph_objects as go

fig = go.Figure()
stages = ['Просмотр (View)', 'Корзина (Cart)', 'Покупка (Purchase)']

for segment in funnel_segmented.index:
    fig.add_trace(go.Funnel(
        name=segment,
        y=stages,
        x=funnel_segmented.loc[segment, ['view', 'cart', 'purchase']].values,
        textinfo="value+percent initial",
        opacity=0.85
    ))

fig.update_layout(
    title="Сегментация воронки конверсии: День vs Ночь",
    title_x=0.5,
    template="plotly_white",
    width=800,
    height=500
)
fig.show()

Выводы по графику воронки: 

Дневная аудитория генерирует более высокую макро-конверсию (6% против 6%) и больше просмотров объявлений почти вдвое больше оплаченных заказов, чем ночная (263,5 тыс. просмотров против 142,6 тыс.):

1. Конверсия добавления в корзину не зависит от времени суток - 28% днем и 29% ночью.

2. Основная потеря ночного сегмента происходит при переходе к чекауту. Ночные покупатели чаще используют корзину как вишлист или откладывают транзакцию до утра.  

3. Максимальный отток происходит на этапе перехода из корзины к оплате Cart -> Purchase.

Можно сформулировать несколько гипотез, почему так происходит: 

Появление стоимости доставки, сервисных сборов или минимального порога суммы заказа только на финальном экране;

Принудительная регистрация и отсутствие опции быстрой гостевой покупки; 

Отсутствие в приложении способов быстрой оплаты вроде СБП, Yandex Pay;

Паттерн поведения пользователей - корзина как вишлист.

Продуктовая рекомендация: Для снижения ночного дроп оффа предлагается внедрить автоматические утренние пуш-уведомления с напоминанием о товарах, добавленных, но не оформленных товаров в корзине. Альтернатива: индивидуальная небольшая скидка, действующая только в ночные часы и ограниченная по времени для стимуляции немедленной оплаты.